# Download Perch v2 ONNX Model

HuggingFace (`justinchuby/Perch-onnx`) から Perch v2 の ONNX モデルをダウンロードし、
Kaggle notebook の output に保存する。

output を Kaggle Model / Dataset としてアップロードして推論ノートブックから参照する。

In [ ]:
!pip install -q huggingface_hub

In [ ]:
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

REPO_ID = "justinchuby/Perch-onnx"
OUT_DIR = Path("/kaggle/working/perch-v2-onnx")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- 1. ONNX model (409 MB) ---
print("Downloading perch_v2.onnx ...")
onnx_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="perch_v2.onnx",
    local_dir=str(OUT_DIR),
)
print(f"  -> {onnx_path}  ({os.path.getsize(onnx_path) / 1e6:.1f} MB)")

print("\nDone!")
print("Files in output:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# --- 2. Quick sanity check ---
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession(str(OUT_DIR / "perch_v2.onnx"))

print("Inputs:")
for inp in sess.get_inputs():
    print(f"  {inp.name}: {inp.shape} ({inp.type})")

print("\nOutputs:")
for out in sess.get_outputs():
    print(f"  {out.name}: {out.shape} ({out.type})")

# Test with 5 seconds of silence (32kHz * 5 = 160000)
dummy = np.zeros((1, 160000), dtype=np.float32)
results = sess.run(None, {"inputs": dummy})

output_names = [o.name for o in sess.get_outputs()]
for name, arr in zip(output_names, results):
    print(f"  {name}: shape={arr.shape}, dtype={arr.dtype}, min={arr.min():.4f}, max={arr.max():.4f}")

print("\nSanity check passed!")